# Genotype PCA

This notebook computes **principal components (PCs) from the genotype data** to use as covariates that correct for population structure in downstream QTL analysis.


## Description

This notebook computes **principal components (PCs)** from the genotype data to use as covariates that correct for population structure in downstream QTL analysis.

To estimate genotype PCs accurately we account for **relatedness** between samples, because closely related individuals share large genomic segments that distort the principal-component space. We use **KING** to estimate kinship and, when relatives exist, split the samples into an **unrelated** set and a **related** set. The PCA model is fit on the unrelated set, and the related samples are then **projected** onto that model so every sample ends up with PC scores in the same space.

> **Note on this toy dataset:** the 60-sample example contains **one related pair** (SAMPLE_059 and SAMPLE_060 are a 1st-degree, parent-offspring pair with kinship ~0.25). KING therefore splits the cohort into **59 unrelated** samples and **1 related** sample (SAMPLE_060). Steps 3-4 fit the PCA model on the 59 unrelated samples; Steps 5-6 project SAMPLE_060 onto that model.

## Input Files

| File | Description |
| --- | --- |
| `output/plink/protocol_example.genotype.merged.plink_qc.{bed,bim,fam}` | QC'd, merged genotype in PLINK format (from `2_genotype_preprocessing.ipynb`). |
| `output/rnaseq/protocol_example.rnaseq.bed.bed.gz` | Phenotype bed (from `1_phenotype_preprocessing.ipynb`); used to define the set of samples that have both genotype and phenotype. |


## Steps


### **Step 1.** Match genotype samples to phenotype samples

Keep only the samples present in **both** the genotype and the phenotype. The resulting `*.sample_genotypes.txt` is the keep-list used by KING in the next step.


In [ ]:
sos run pipeline/GWAS_QC.ipynb genotype_phenotype_sample_overlap \
    --cwd output/genotype/ \
    --genoFile output/plink/protocol_example.genotype.merged.plink_qc.fam \
    --phenoFile output/rnaseq/protocol_example.rnaseq.bed.bed.gz


### **Step 2.** Estimate kinship (KING)

KING estimates pairwise kinship coefficients and, if relatives are found, splits the cohort into an **unrelated** set (`*.king.unrelated.bed`) and a **related** set (`*.king.related.bed`). The `.kin0` file lists kinship coefficients for related pairs.

On this toy dataset KING detects the related pair **SAMPLE_059 - SAMPLE_060** (kinship ~0.25, IBS0 = 0, i.e. parent-offspring). It writes `*.king.related.bed` (1 sample: SAMPLE_060) and `*.king.unrelated.bed` (59 samples), and we proceed with the unrelated set in Step 3.

In [ ]:
sos run pipeline/GWAS_QC.ipynb king \
    --cwd output/genotype/kinship \
    --genoFile output/plink/protocol_example.genotype.merged.plink_qc.bed \
    --name protocol_example.genotype.king \
    --keep-samples output/genotype/protocol_example.rnaseq.bed.bed.sample_genotypes.txt


### **Step 3.** QC + LD pruning on the unrelated set

We apply variant- and sample-level QC (here filtering minor-allele count < 5) and **LD-prune the 59 unrelated samples** (`*.king.unrelated.bed`) in preparation for PCA. LD pruning removes highly correlated SNPs so the PCs reflect genome-wide structure. The pruned set (`*.prune.bed`) and retained-SNP list (`*.prune.in`) are written under the working directory.

> **LD-pruning sample-size requirement:** plink2 `--indep-pairwise` LD estimator **requires at least 50 samples**; with 59 unrelated samples this runs cleanly.

In [ ]:
# QC + LD pruning on the 59 unrelated samples (KING unrelated set)
sos run pipeline/GWAS_QC.ipynb qc \
    --cwd output/genotype/genotype \
    --genoFile output/genotype/kinship/protocol_example.genotype.merged.plink_qc.protocol_example.genotype.king.unrelated.bed \
    --mac-filter 5 -s force

### **Step 4.** flashPCA on the unrelated set

Run `flashpca` on the pruned unrelated genotype. This fits the PCA model, computes Mahalanobis distances to flag outliers, and plots PC scatter and scree plot. These genotype PCs are the population-structure covariates used in the QTL analysis.

In [ ]:
sos run pipeline/PCA.ipynb flashpca \
    --cwd output/genotype/genotype_pca \
    --genoFile output/genotype/genotype/protocol_example.genotype.merged.plink_qc.protocol_example.genotype.king.unrelated.plink_qc.prune.bed

### **Step 5.** QC the related set (keep unrelated-PCA SNPs)

QC the related samples **without** LD pruning, restricted to the same SNPs retained for the unrelated PCA (`--keep-variants ...prune.in`), so the related genotypes are on the identical SNP set as the PCA model.

In [ ]:
K=output/genotype/kinship/protocol_example.genotype.merged.plink_qc.protocol_example.genotype.king
sos run pipeline/GWAS_QC.ipynb qc_no_prune \
    --cwd output/genotype/genotype \
    --genoFile ${K}.related.bed \
    --maf-filter 0 --geno-filter 0 --mind-filter 0.1 --hwe-filter 0 \
    --keep-variants output/genotype/genotype/protocol_example.genotype.merged.plink_qc.protocol_example.genotype.king.unrelated.plink_qc.prune.in

### **Step 6.** Project the related sample onto the unrelated PCA model

Project the related sample (SAMPLE_060) onto the unrelated PCA model with `PCA.ipynb project_samples`, so every sample ends up with PC scores in the same space.

In [ ]:
sos run pipeline/PCA.ipynb project_samples \
    --cwd output/genotype/genotype_pca \
    --genoFile output/genotype/genotype/protocol_example.genotype.merged.plink_qc.protocol_example.genotype.king.related.plink_qc.extracted.bed \
    --pca-model output/genotype/genotype_pca/protocol_example.genotype.merged.plink_qc.protocol_example.genotype.king.unrelated.plink_qc.prune.pca.rds

The final pca output that we will use in cov processing
`related.plink_qc.extracted.pca.projected.rds`

## Output Files

| File | Description |
| --- | --- |
| `output/genotype/protocol_example.rnaseq.bed.bed.sample_genotypes.txt` | Samples present in both genotype and phenotype (keep-list). |
| `output/genotype/kinship/*.king.kin0` | KING kinship table; lists the related pair SAMPLE_059 - SAMPLE_060. |
| `output/genotype/kinship/*.king.unrelated.bed` / `*.king.related.bed` | Genotype split into 59 unrelated and 1 related (SAMPLE_060) samples. |
| `output/genotype/genotype/*.king.unrelated.plink_qc.prune.{bed,in}` | QC + LD-pruned unrelated genotype and the retained-SNP list. |
| `output/genotype/genotype_pca/*.king.unrelated.plink_qc.prune.pca.rds` | Fitted flashPCA model (PCs for the 59 unrelated samples). |
| `output/genotype/genotype/*.king.related.plink_qc.extracted.bed` | Related sample restricted to the unrelated-PCA SNP set. |
| `output/genotype/genotype_pca/*.extracted.pca.projected.*` | Related sample (SAMPLE_060) projected onto the unrelated PCA model (`.pc.png`, `.scree.png`, `.mahalanobis.rds`). |

## Anticipated Results

KING detects **one related pair** among the 60 samples (SAMPLE_059 - SAMPLE_060, kinship ~0.25, IBS0 = 0 -> parent-offspring) and splits the cohort into **59 unrelated** and **1 related** sample (SAMPLE_060). QC + LD pruning and flashPCA run on the **59 unrelated samples** to fit the PCA model (`*.unrelated.plink_qc.prune.pca.rds`), producing PC scatter / scree plots and a Mahalanobis outlier table. The related sample is then QC-restricted to the unrelated-PCA SNP set and **projected** onto that model (`*.extracted.pca.projected.*`), so all 60 samples have genotype PCs in the same space for use as covariates in the QTL analysis.